In [0]:
%run ./config

✅ Config loaded


In [0]:


# ============================================
# CONFIGURATION - CHANGE FOR TEST/PROD
# ============================================

# Set to True for testing, False for production
TEST_MODE = True  # ← Change to False for production

# Storage configuration
STORAGE_ACCOUNT = "funddatalakeshantanu"
CONTAINER_NAME = "bronze"

# Volume path - automatically switches based on TEST_MODE
if TEST_MODE:
    VOLUME_PATH = "/Volumes/workspace/default/test_ingest_volume/"
    print("🧪 TEST MODE ENABLED")
else:
    VOLUME_PATH = "/Volumes/workspace/default/adls_ingest_volume/"
    print("🏭 PRODUCTION MODE ENABLED")

print(f"✅ Using volume: {VOLUME_PATH}")
print("✅ Credentials loaded from config notebook")

# ============================================
# DOWNLOAD FILES
# ============================================

import os
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential

# Create credential object
credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)

# Connect to ADLS
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential
)

print("✅ Connected to ADLS")

# Get files from container
file_system_client = service_client.get_file_system_client(file_system=CONTAINER_NAME)
paths = file_system_client.get_paths()

print("📥 Downloading files...")
count = 0

for path in paths:
    if not path.is_directory:
        file_client = file_system_client.get_file_client(path.name)
        download = file_client.download_file()
        file_content = download.readall()
        
        local_file_name = os.path.join(VOLUME_PATH, os.path.basename(path.name))
        with open(local_file_name, 'wb') as local_file:
            local_file.write(file_content)
        
        count += 1
        print(f"✅ Downloaded: {path.name}")

print(f"\n🎉 Downloaded {count} file(s) to Volume")

# ============================================
# VERIFY
# ============================================

print("\n📂 Files in Volume:")
display(dbutils.fs.ls(VOLUME_PATH))

🧪 TEST MODE ENABLED
✅ Using volume: /Volumes/workspace/default/test_ingest_volume/
✅ Credentials loaded from config notebook
✅ Connected to ADLS
📥 Downloading files...
✅ Downloaded: AAPL_17cd89cc-f7dc-4fe8-b9a6-c95b7766750c.json
✅ Downloaded: GOOG_c5ce959e-f2ea-4264-81d8-39b630ae5b9e.json
✅ Downloaded: MSFT_ca5bda0b-890f-471e-ba7b-b18fcc8c514b.json
✅ Downloaded: NVDA_005b0806-03e0-4eac-bcb8-6b8b77e0397a.json
✅ Downloaded: TSLA_b6c6920e-8d48-484c-8708-944b8aff3069.json

🎉 Downloaded 5 file(s) to Volume

📂 Files in Volume:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/test_ingest_volume/AAPL_17cd89cc-f7dc-4fe8-b9a6-c95b7766750c.json,AAPL_17cd89cc-f7dc-4fe8-b9a6-c95b7766750c.json,36706,1785687613000
dbfs:/Volumes/workspace/default/test_ingest_volume/GOOG_c5ce959e-f2ea-4264-81d8-39b630ae5b9e.json,GOOG_c5ce959e-f2ea-4264-81d8-39b630ae5b9e.json,36123,1785687613000
dbfs:/Volumes/workspace/default/test_ingest_volume/MSFT_ca5bda0b-890f-471e-ba7b-b18fcc8c514b.json,MSFT_ca5bda0b-890f-471e-ba7b-b18fcc8c514b.json,36204,1785687613000
dbfs:/Volumes/workspace/default/test_ingest_volume/NVDA_005b0806-03e0-4eac-bcb8-6b8b77e0397a.json,NVDA_005b0806-03e0-4eac-bcb8-6b8b77e0397a.json,36953,1785687613000
dbfs:/Volumes/workspace/default/test_ingest_volume/TSLA_b6c6920e-8d48-484c-8708-944b8aff3069.json,TSLA_b6c6920e-8d48-484c-8708-944b8aff3069.json,36229,1785687613000
